# Visual Context Models: Monocular Depth & Anomaly Detection

This repository explores the application of state-of-the-art vision foundation models for two complex computer vision tasks:
1. **Monocular Depth Estimation** using `Depth-Anything-V2`.
2. **Unsupervised Anomaly Detection** using zero-shot features extracted from `DINOv2`.

Instead of training models from scratch, we leverage the deep semantic understanding of pre-trained models to solve these tasks.

In [ ]:
import os
import time
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForDepthEstimation, AutoModel
from sklearn.metrics import roc_auc_score
import kagglehub

# Setting up the computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {device}")


## Part 1: Monocular Depth Estimation

Monocular depth models estimate the distance of objects in a scene from a single 2D RGB image. Because 2D images inherently lack physical depth context, these models suffer from scale ambiguity. To evaluate them against hardware sensor ground-truth (like the NYU-Depth-v2 dataset), we apply **median scale alignment** to match the real-world physical scale.

In this section, we use `Depth-Anything-V2-Small` to estimate depth and evaluate it using Mean Absolute Relative Error (AbsRel).

In [ ]:
print("Loading NYU-Depth-v2 (Validation Subset)...")
nyu_val = load_dataset("tanganke/nyuv2", split="val")
nyu_subset = nyu_val.select(range(100)) # Using 100 images for evaluation

model_name = "depth-anything/Depth-Anything-V2-Small-hf"
depth_processor = AutoImageProcessor.from_pretrained(model_name, use_fast=False)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_name).to(device)
depth_model.eval()

abs_rel_scores = []
total_inference_time = 0.0

print("Running evaluation...")
for item in tqdm(nyu_subset):
    image_data = item['image']
    gt_depth = np.array(item['depth'])
    if len(gt_depth.shape) == 3:
        gt_depth = gt_depth.squeeze(0)

    image_np = np.array(image_data)
    image_pil = Image.fromarray(image_np.transpose(1, 2, 0).astype(np.uint8))

    inputs = depth_processor(images=image_pil, return_tensors="pt").to(device)

    start_time = time.time()
    with torch.no_grad():
        outputs = depth_model(**inputs)
    total_inference_time += (time.time() - start_time)

    # Depth-Anything predicts inverse depth (disparity)
    predicted_inverse_depth = outputs.predicted_depth.unsqueeze(1)

    upsampled_pred_inv = F.interpolate(
        predicted_inverse_depth,
        size=gt_depth.shape,
        mode="bicubic",
        align_corners=False
    ).squeeze().cpu().numpy()

    # Convert inverse depth to actual depth
    pred_depth_np = 1.0 / (upsampled_pred_inv + 1e-8)

    # Only calculate metrics on valid sensor pixels
    valid_mask = gt_depth > 0
    gt_valid = gt_depth[valid_mask]
    pred_valid = pred_depth_np[valid_mask]

    # Median scale alignment to resolve scale ambiguity
    s = np.median(gt_valid) / (np.median(pred_valid) + 1e-8)
    aligned_pred_valid = pred_valid * s

    # Calculate AbsRel
    abs_rel = np.mean(np.abs(aligned_pred_valid - gt_valid) / gt_valid)
    abs_rel_scores.append(abs_rel)

print(f"Total Inference Time: {total_inference_time:.4f} seconds")
print(f"Runtime per image: {total_inference_time / len(nyu_subset):.4f} seconds")
print(f"Mean AbsRel Error: {np.mean(abs_rel_scores):.4f}")

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 20))
plt.suptitle("Monocular Depth Estimation: RGB vs Pred vs Ground Truth", fontsize=16)

for i in range(5):
    item = nyu_subset[i]
    raw_image = item['image']
    image_pil = raw_image.convert("RGB") if isinstance(raw_image, Image.Image) else Image.fromarray(np.transpose(np.array(raw_image), (1, 2, 0)).astype(np.uint8)).convert("RGB")

    gt_depth = np.array(item['depth']).squeeze(0) if len(np.array(item['depth']).shape) == 3 else np.array(item['depth'])

    inputs = depth_processor(images=image_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = depth_model(**inputs)

    upsampled_pred_inv = F.interpolate(outputs.predicted_depth.unsqueeze(1), size=gt_depth.shape, mode="bicubic", align_corners=False).squeeze().cpu().numpy()
    pred_depth_np = 1.0 / (upsampled_pred_inv + 1e-8)

    valid_mask = gt_depth > 0
    s = np.median(gt_depth[valid_mask]) / (np.median(pred_depth_np[valid_mask]) + 1e-8) if valid_mask.any() else 1.0
    aligned_pred = pred_depth_np * s

    row_min = min(np.min(aligned_pred[valid_mask]), np.min(gt_depth[valid_mask]))
    row_max = max(np.max(aligned_pred[valid_mask]), np.max(gt_depth[valid_mask]))

    axes[i, 0].imshow(image_pil)
    axes[i, 0].set_title("RGB Input")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(aligned_pred, cmap='magma', vmin=row_min, vmax=row_max)
    axes[i, 1].set_title("Aligned Predicted Depth")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(np.where(valid_mask, gt_depth, np.nan), cmap='magma', vmin=row_min, vmax=row_max)
    axes[i, 2].set_title("Hardware Ground Truth")
    axes[i, 2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

## Part 2: Unsupervised Anomaly Detection with DINOv2

Traditional anomaly detection requires thousands of labeled defect examples. Here, we build an entirely unsupervised pipeline using the **MVTec-AD dataset**.

We use `DINOv2`, a self-supervised vision transformer, as a frozen feature extractor. We build a "Memory Bank" of patch embeddings from *only nominal (good) images*. During testing, we extract features from unseen images and flag anomalies by calculating the maximum nearest-neighbor distance of its patches against the memory bank. If a structural defect exists, its semantic embedding will stray far from the nominal cluster.

In [ ]:
print("Downloading MVTec-AD...")
mvtec_path = kagglehub.dataset_download("ipythonx/mvtec-ad")
categories = ["leather", "bottle", "transistor"]

dino_model_name = "facebook/dinov2-small"
dino_processor = AutoImageProcessor.from_pretrained(dino_model_name)
dino_model = AutoModel.from_pretrained(dino_model_name).to(device)
dino_model.eval()

def extract_features(image_path):
    """Extracts and L2-normalizes DINOv2 patch tokens."""
    img = Image.open(image_path).convert("RGB")
    original_size = img.size[::-1]
    inputs = dino_processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = dino_model(**inputs)

    # Ignore [CLS] token, take spatial patches
    patch_tokens = outputs.last_hidden_state[:, 1:, :]
    patch_tokens = F.normalize(patch_tokens, p=2, dim=-1)

    grid_h = inputs['pixel_values'].shape[-2] // 14
    grid_w = inputs['pixel_values'].shape[-1] // 14

    return patch_tokens.squeeze(0), grid_h, grid_w, original_size

In [ ]:
aurocs = []
vis_data = {}

for cat in categories:
    print(f"\nProcessing: {cat.capitalize()}")
    cat_dir = os.path.join(mvtec_path, cat)

    # 1. Build Memory Bank from Good/Nominal Train Images
    train_imgs = glob.glob(f"{cat_dir}/train/good/*.png")
    memory_bank = []
    for img_path in tqdm(train_imgs, desc="Building Memory Bank"):
        features, _, _, _ = extract_features(img_path)
        memory_bank.append(features)

    memory_bank = torch.cat(memory_bank, dim=0)

    # 2. Score Test Images
    test_imgs = glob.glob(f"{cat_dir}/test/*/*.png")
    image_scores, labels = [], []
    saved_vis = {'good': [], 'defective': []}

    for img_path in tqdm(test_imgs, desc="Scoring Test Set"):
        is_defective = "/test/good/" not in img_path
        labels.append(1 if is_defective else 0)

        test_features, grid_h, grid_w, orig_size = extract_features(img_path)

        # Calculate distances to memory bank
        similarity = torch.mm(test_features, memory_bank.T)
        distances = 1 - similarity
        min_patch_distances, _ = torch.min(distances, dim=1)

        # Image anomaly score is the max of patch anomaly scores
        image_scores.append(torch.max(min_patch_distances).item())

        # Save specific images for visualization
        if (is_defective and len(saved_vis['defective']) < 2) or (not is_defective and len(saved_vis['good']) < 1):
            heatmap = min_patch_distances.view(1, 1, grid_h, grid_w)
            heatmap_np = F.interpolate(heatmap, size=orig_size, mode='bilinear', align_corners=False).squeeze().cpu().numpy()

            gt_mask_path = None
            if is_defective:
                defect_type = os.path.basename(os.path.dirname(img_path))
                mask_name = os.path.basename(img_path).replace(".png", "_mask.png")
                gt_mask_path = os.path.join(cat_dir, "ground_truth", defect_type, mask_name)

            group = 'defective' if is_defective else 'good'
            saved_vis[group].append({'img_path': img_path, 'heatmap': heatmap_np, 'gt_mask_path': gt_mask_path})

    auroc = roc_auc_score(labels, image_scores)
    aurocs.append(auroc)
    print(f"AUROC: {auroc:.4f}")
    vis_data[cat] = saved_vis['defective'] + saved_vis['good']

print(f"\nMean AUROC across categories: {np.mean(aurocs):.4f}")

In [ ]:
fig, axes = plt.subplots(3 * 3, 3, figsize=(12, 30))
plt.suptitle("MVTec-AD Anomaly Detection: DINOv2 Heatmaps", fontsize=16)

row_idx = 0
for cat in categories:
    for ex in vis_data[cat]:
        img_np = np.array(Image.open(ex['img_path']).convert("RGB"))

        # Normalize and colorize heatmap
        heatmap = ex['heatmap']
        heatmap_norm = ((heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8) * 255).astype(np.uint8)
        heatmap_color = cv2.cvtColor(cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_np, 0.5, heatmap_color, 0.5, 0)

        # Load GT
        gt_mask = np.array(Image.open(ex['gt_mask_path']).convert("L")) if ex['gt_mask_path'] and os.path.exists(ex['gt_mask_path']) else np.zeros(img_np.shape[:2], dtype=np.uint8)

        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title(f"{cat.capitalize()} - {'Defective' if ex['gt_mask_path'] else 'Nominal'}")
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(overlay)
        axes[row_idx, 1].set_title("Distance Heatmap")
        axes[row_idx, 1].axis('off')

        axes[row_idx, 2].imshow(gt_mask, cmap='gray')
        axes[row_idx, 2].set_title("Ground Truth Defect Mask")
        axes[row_idx, 2].axis('off')

        row_idx += 1

plt.tight_layout(rect=[0, 0.02, 1, 0.98])
plt.show()